In [4]:
import pandas as pd

df = pd.read_csv('data/output/combined.csv')

In [5]:
import pandas as pd

# --- DROP UNUSED / REDUNDANT COLUMNS ---
drop_cols = [
    'Locality',
    'REF_DATE',
    'Coordinate',
    'Geography',
    'GEO_LEVEL',
    'CENSUS_YEAR',
    'GEO_NAME',
    'Private dwellings occupied by usual residents, 2016',
    'Private dwellings occupied by usual residents, 2021',
    'Total private dwellings, 2016',
    'Total private dwellings, 2021',
    'Age',
    'Gender',
    'Statistics'
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# --- RENAME COLUMNS ---
df = df.rename(columns={
    'Average Unmployment Rate (2021)': 'average_unemployment_rate_2021',
    'Province': 'province',
    'Population and dwelling counts (13): Land area in square kilometres, 2021 [10]': 'land_area_km2_2021',

    # official languages
    'English and French': 'knowledge_of_official_languages_english_and_french',
    'English only': 'knowledge_of_official_languages_english_only',
    'French only': 'knowledge_of_official_languages_french_only',
    'Neither English nor French': 'knowledge_of_official_languages_neither',
    'Total - Knowledge of official languages': 'knowledge_of_official_languages_total',

    # age groups
    '0 to 14 years': 'age_0_14_years',
    '15 to 64 years': 'age_15_64_years',
    '65 years and over': 'age_65_plus_years',

    # population stats
    'Land area in square kilometres, 2021': 'land_area_km2_2021_alt',
    'National population rank, 2021': 'national_population_rank_2021',
    'Population density per square kilometre, 2021': 'population_density_km2_2021',
    'Population percentage change, 2016 to 2021': 'population_pct_change_2016_2021',
    'Population, 2016': 'population_2016',
    'Population, 2021': 'population_2021',
    'Private dwellings occupied by usual residents percentage change, 2016 to 2021': 'private_dwellings_occupied_pct_change_2016_2021',
    'Province/territory population rank, 2021': 'province_population_rank_2021',
    'Total private dwellings percentage change, 2016 to 2021': 'total_private_dwellings_pct_change_2016_2021',

    # immigration
    'Total': 'total_population',
    'Non_Immigrants': 'non_immigrants_total',
    'Immigrants': 'immigrants_total',
    'Recent_Immigrants_2016_2021': 'recent_immigrants_2016_2021_total',
    'Non_Permanent_Residents': 'non_permanent_residents_total',

    # minorities
    'Total - Minority': 'minority_population_pct',
    'Total Minority Population': 'minority_population_total',
    'South Asian': 'minority_south_asian',
    'Chinese': 'minority_chinese',
    'Black': 'minority_black',
    'Filipino': 'minority_filipino',
    'Arab': 'minority_arab',
    'Latin American': 'minority_latin_american',
    'Southeast Asian': 'minority_southeast_asian',
    'West Asian': 'minority_west_asian',
    'Korean': 'minority_korean',
    'Japanese': 'minority_japanese',
    'Minority, n.i.e': 'minority_not_included_elsewhere',
    'Multiple Minorities': 'minority_multiple',
    'Not Minority': 'not_minority_population'
})

# --- STANDARDIZE PROVINCE CODES ---
province_map = {
    'Ontario': 'ON', 'Quebec': 'QC', 'British Columbia': 'BC', 'Alberta': 'AB',
    'Manitoba': 'MB', 'Saskatchewan': 'SK', 'Nova Scotia': 'NS',
    'New Brunswick': 'NB', 'Newfoundland and Labrador': 'NL',
    'Prince Edward Island': 'PE', 'Northwest Territories': 'NT',
    'Yukon': 'YT', 'Nunavut': 'NU'
}

df['province'] = df['province'].replace(province_map)

# --- OPTIONAL: CONSOLIDATE MINORITY COLUMNS INTO ONE STRUCTURED COLUMN ---
minority_cols = [
    'minority_south_asian',
    'minority_chinese',
    'minority_black',
    'minority_filipino',
    'minority_arab',
    'minority_latin_american',
    'minority_southeast_asian',
    'minority_west_asian',
    'minority_korean',
    'minority_japanese',
    'minority_not_included_elsewhere',
    'minority_multiple',
    'not_minority_population'
]

def build_minority_dict(row):
    return {col.replace('minority_', ''): row[col] for col in minority_cols if col in row}

df['minority_breakdown'] = df.apply(build_minority_dict, axis=1)

# OPTIONAL: drop original columns after consolidation
# df = df.drop(columns=minority_cols)

# --- CLEANUP DUPLICATE LAND AREA COLUMN ---
if 'land_area_km2_2021_alt' in df.columns:
    df['land_area_km2_2021'] = df['land_area_km2_2021'].fillna(df['land_area_km2_2021_alt'])
    df = df.drop(columns=['land_area_km2_2021_alt'])